In [5]:
# =============================================
# SMART MANUFACTURING AI SYSTEM
# Dataset: Kaggle AI4I 2020 (10,000 rows)
# =============================================

# Import required libraries
import kagglehub
import pandas as pd

# Download dataset from Kaggle
path = kagglehub.dataset_download("stephanmatzka/predictive-maintenance-dataset-ai4i-2020")

# Load CSV file into dataframe
df = pd.read_csv(path + "/ai4i2020.csv")

# Rename columns for easier use
df = df.rename(columns={
    'Air temperature [K]':     'air_temp',   # Air temperature
    'Process temperature [K]': 'proc_temp',  # Process temperature
    'Rotational speed [rpm]':  'rpm',        # Machine speed
    'Torque [Nm]':             'torque',     # Machine torque
    'Tool wear [min]':         'tool_wear',  # Tool wear time
    'Machine failure':         'failure'     # 0=Good, 1=Fail
})

print(f"Data ready! Shape: {df.shape}")

Using Colab cache for faster access to the 'predictive-maintenance-dataset-ai4i-2020' dataset.
Data ready! Shape: (10000, 14)


In [6]:
# =============================================
# SECTION 1: STATISTICS
# Understanding the data with numbers
# =============================================

print("=== SECTION 1: STATISTICS ===")

# Basic statistics: mean, min, max, std deviation
print(df[['air_temp','proc_temp','rpm','torque','tool_wear','failure']].describe())

# Count total machine failures
print(f"\nTotal Failures: {df['failure'].sum()} / {len(df)}")

# Check correlation of each feature with failure
print(f"\nCorrelations with failure:")
print(df[['air_temp','proc_temp','rpm','torque','tool_wear','failure']].corr()['failure'])

=== SECTION 1: STATISTICS ===
           air_temp     proc_temp           rpm        torque     tool_wear  \
count  10000.000000  10000.000000  10000.000000  10000.000000  10000.000000   
mean     300.004930    310.005560   1538.776100     39.986910    107.951000   
std        2.000259      1.483734    179.284096      9.968934     63.654147   
min      295.300000    305.700000   1168.000000      3.800000      0.000000   
25%      298.300000    308.800000   1423.000000     33.200000     53.000000   
50%      300.100000    310.100000   1503.000000     40.100000    108.000000   
75%      301.500000    311.100000   1612.000000     46.800000    162.000000   
max      304.500000    313.800000   2886.000000     76.600000    253.000000   

            failure  
count  10000.000000  
mean       0.033900  
std        0.180981  
min        0.000000  
25%        0.000000  
50%        0.000000  
75%        0.000000  
max        1.000000  

Total Failures: 339 / 10000

Correlations with failure:
air

In [7]:
# =============================================
# SECTION 2: LINEAR REGRESSION
# Predict Torque from RPM
# =============================================

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np

print("=== SECTION 2: LINEAR REGRESSION ===")

# Define input (RPM) and output (Torque)
X = df[['rpm']].values
y = df['torque'].values

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate model performance
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")

# Test with example values
print(f"RPM 1000 → Torque: {model.predict([[1000]])[0]:.1f} Nm")
print(f"RPM 2000 → Torque: {model.predict([[2000]])[0]:.1f} Nm")

=== SECTION 2: LINEAR REGRESSION ===
R² Score: 0.757
RPM 1000 → Torque: 66.1 Nm
RPM 2000 → Torque: 17.7 Nm


In [8]:
# =============================================
# SECTION 3: LOGISTIC REGRESSION
# Predict machine failure (Yes or No)
# =============================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("=== SECTION 3: LOGISTIC REGRESSION ===")

# Use 5 features to predict failure
X = df[['air_temp','proc_temp','rpm','torque','tool_wear']].values
y = df['failure'].values  # 0=Good, 1=Fail

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Print results
print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%}")
print(classification_report(y_test, y_pred))

=== SECTION 3: LOGISTIC REGRESSION ===
Accuracy: 97.2%
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1939
           1       0.62      0.26      0.37        61

    accuracy                           0.97      2000
   macro avg       0.80      0.63      0.68      2000
weighted avg       0.97      0.97      0.97      2000



In [9]:
# =============================================
# SECTION 4: RANDOM FOREST
# 100 decision trees voting together
# =============================================

from sklearn.ensemble import RandomForestClassifier

print("=== SECTION 4: RANDOM FOREST ===")

X = df[['air_temp','proc_temp','rpm','torque','tool_wear']].values
y = df['failure'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Create forest with 100 trees
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%}")

# Show which features matter most
print("\nFeature Importance:")
features = ['air_temp','proc_temp','rpm','torque','tool_wear']
for f, i in zip(features, model.feature_importances_):
    print(f"  {f}: {i:.1%}")

=== SECTION 4: RANDOM FOREST ===
Accuracy: 98.4%

Feature Importance:
  air_temp: 16.2%
  proc_temp: 14.9%
  rpm: 21.5%
  torque: 32.1%
  tool_wear: 15.4%


In [10]:
# =============================================
# SECTION 5: NEURAL NETWORK
# Multi-layer perceptron for classification
# =============================================

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

print("=== SECTION 5: NEURAL NETWORK ===")

X = df[['air_temp','proc_temp','rpm','torque','tool_wear']].values
y = df['failure'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale features - required for neural networks
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Build network: 64 neurons → 32 neurons → Output
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%}")

=== SECTION 5: NEURAL NETWORK ===
Accuracy: 98.2%


In [11]:
# =============================================
# SECTION 7: NUMERICAL METHODS
# Solving engineering equations numerically
# =============================================

import numpy as np
from scipy.optimize import fsolve

print("=== SECTION 7: NUMERICAL METHODS ===")

# Problem 1: Calculate stress on machine
force  = 20000   # Newtons
area   = 0.001   # Square meters
stress = force / area
print(f"\n1. Stress = {stress:,.0f} Pascal")
print("   WARNING: Stress too high!" if stress > 15000000 else "   Safe")

# Problem 2: Find time to reach 70 degrees
def temp_equation(t):
    # Returns 0 when temperature equals 70
    return 100 * (1 - np.exp(-0.1 * t)) - 70

time = fsolve(temp_equation, x0=10)[0]
print(f"\n2. Time to reach 70C = {time:.1f} minutes")

# Problem 3: Find required pipe radius
def flow_equation(r):
    # Returns 0 when flow rate is correct
    return np.pi * r**2 * 2 - 0.005

radius = fsolve(flow_equation, x0=0.05)[0]
print(f"\n3. Required pipe radius = {radius*100:.1f} cm")

=== SECTION 7: NUMERICAL METHODS ===

1. Stress = 20,000,000 Pascal

2. Time to reach 70C = 12.0 minutes

3. Required pipe radius = 2.8 cm


In [12]:
# =============================================
# SECTION 8: LANGCHAIN
# Using templates to structure AI prompts
# =============================================

from transformers import pipeline
from langchain_core.prompts import PromptTemplate

print("=== SECTION 8: LANGCHAIN ===")

# Load zero-shot classification model
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

# Create a reusable prompt template
template = PromptTemplate(
    input_variables=["machine_status"],
    template="Manufacturing status: {machine_status}"
)

# Machine status messages to classify
statuses = [
    "high temperature and pressure",
    "normal operation",
    "motor vibration detected",
    "low speed, all good",
    "tool wear critical"
]

# Possible categories
labels = ["critical failure", "normal", "warning"]

# Classify each status
for status in statuses:
    prompt = template.format(machine_status=status)
    result = classifier(prompt, labels)
    print(f"{status[:35]:<35} → {result['labels'][0]}")

=== SECTION 8: LANGCHAIN ===


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

high temperature and pressure       → warning
normal operation                    → normal
motor vibration detected            → warning
low speed, all good                 → normal
tool wear critical                  → warning


In [14]:
# =============================================
# SECTION 9: NEUROSYMBOLIC AI
# Combining AI predictions with expert rules
# Neural = AI model, Symbolic = Rule-based logic
# =============================================

print("=== SECTION 9: NEUROSYMBOLIC AI ===")

# Machine data with sensor readings
machines = [
    {"id": "M1", "temp": 85, "rpm": 1500, "tool_wear": 180,
     "status": "high temperature detected"},
    {"id": "M2", "temp": 45, "rpm": 1200, "tool_wear": 50,
     "status": "normal operation running"},
    {"id": "M3", "temp": 72, "rpm": 1800, "tool_wear": 160,
     "status": "vibration and wear detected"},
    {"id": "M4", "temp": 38, "rpm": 1100, "tool_wear": 30,
     "status": "all systems good"},
]

labels = ["dangerous", "safe"]

for machine in machines:
    # Neural part: AI classifies the status text
    result  = classifier(machine["status"], labels)
    ai_says = result['labels'][0]

    # Symbolic part: Check sensor values against rules
    rule_danger = (
        machine["temp"]      > 80  or  # Temperature limit
        machine["tool_wear"] > 150 or  # Wear limit
        machine["rpm"]       > 1700    # Speed limit
    )

    # Final decision: danger if AI or rules say so
    if ai_says == "dangerous" or rule_danger:
        final = "STOP MACHINE ⚠️"
    else:
        final = "CONTINUE ✅"

    print(f"Machine {machine['id']}: "
          f"AI={ai_says} | "
          f"Rule={'Danger' if rule_danger else 'OK'} | "
          f"{final}")

=== SECTION 9: NEUROSYMBOLIC AI ===
Machine M1: AI=dangerous | Rule=Danger | STOP MACHINE ⚠️
Machine M2: AI=safe | Rule=OK | CONTINUE ✅
Machine M3: AI=safe | Rule=Danger | STOP MACHINE ⚠️
Machine M4: AI=safe | Rule=OK | CONTINUE ✅


In [15]:
# =============================================
# SECTION 10: CLIP
# Matching images with text descriptions
# OpenAI model that understands both
# =============================================

from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel
from io import BytesIO

print("=== SECTION 10: CLIP ===")

# Load CLIP model and processor
model_clip     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Download a sample image from internet
url      = "https://images.unsplash.com/photo-1495474472287-4d71bcdd2085?w=400"
response = requests.get(url)
image    = Image.open(BytesIO(response.content)).convert("RGB")

# Text labels to compare with image
texts = [
    "a working machine",
    "a broken machine",
    "a cup of coffee",
    "industrial equipment"
]

# Process image and text together
inputs  = processor_clip(text=texts, images=image,
                         return_tensors="pt", padding=True)
outputs = model_clip(**inputs)

# Convert to percentages
probs = outputs.logits_per_image.softmax(dim=1)

# Print similarity scores
for text, prob in zip(texts, probs[0]):
    print(f"{text:<30} → {prob:.0%}")

=== SECTION 10: CLIP ===


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (4 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

a working machine              → 0%
a broken machine               → 0%
a cup of coffee                → 100%
industrial equipment           → 0%
